# Bulk RNA-seq Preprocessing

Convert bulk RNA-seq data (`expression.h5`, `gene_list.txt`, `metadata.csv`) into the
CancerFoundation pipeline format (`vocab.json`, `obs.parquet`, `mapping.json`, `mem.map/`).

In [ ]:
# Inputs and general settings
from pathlib import Path

# --- INPUT: bulk data files (in the same directory) ---
BULK_DIR = Path("/cluster/work/boeva/bulkFM/data/processed/archs4")
EXPRESSION_H5 = BULK_DIR / "expression.h5"
GENE_LIST_TXT = BULK_DIR / "gene_list.txt"
METADATA_CSV = BULK_DIR / "metadata.csv"

# --- OUTPUT: pipeline-ready processed data ---
OUTPUT_DIR = Path("/cluster/work/boeva/bulkFM/data/processed/archs4/pipeline")

# Optional: path to an existing vocab.json to reuse (set to None to generate from gene_list.txt)
EXISTING_VOCAB_PATH = None

# Metadata columns to include in obs.parquet (must exist in metadata.csv)
OBS_COLUMNS = ["tissue", "series_id"]

# How many samples per h5ad chunk (controls memory during conversion)
CHUNK_SIZE = 5000

['expression']


In [ ]:
## Imports
import sys
sys.path.insert(0, "./")

import h5py
import json
import os
import shutil
import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse as sp
from bionemo.scdl.io.single_cell_collection import SingleCellCollection

from cancerfoundation.data.dataset import DatasetDir

GENE_ID = "_cf_gene_id"
CLS_TOKEN = "<cls>"
PAD_TOKEN = "<pad>"

## 1. Load bulk data

In [ ]:
# Load gene list
with open(GENE_LIST_TXT) as f:
    gene_names = [line.strip() for line in f if line.strip()]

print(f"Genes: {len(gene_names)}")

# Load metadata
metadata = pd.read_csv(METADATA_CSV)
print(f"Samples in metadata: {len(metadata)}")
print(f"Columns: {list(metadata.columns)}")
metadata.head()

In [ ]:
# Inspect expression matrix shape without loading it all
with h5py.File(EXPRESSION_H5, "r") as f:
    print("Datasets in HDF5:", list(f.keys()))
    # Adapt the dataset key if needed (common keys: "expression", "data/expression", "X")
    for key in f.keys():
        ds = f[key]
        if hasattr(ds, "shape"):
            print(f"  {key}: shape={ds.shape}, dtype={ds.dtype}")

    # Determine the expression dataset key
    expr_key = "expression"  # adjust if your file uses a different key
    n_samples, n_genes_h5 = f[expr_key].shape
    print(f"\nExpression matrix: {n_samples} samples × {n_genes_h5} genes")
    assert n_genes_h5 == len(gene_names), (
        f"Gene count mismatch: h5 has {n_genes_h5}, gene_list.txt has {len(gene_names)}"
    )

## 2. Build vocabulary

Same format as the single-cell pipeline: `{"<cls>": 0, "<pad>": 1, "GENE_A": 2, ...}`

In [ ]:
data_path = DatasetDir(OUTPUT_DIR)
data_path.mkdir()

if EXISTING_VOCAB_PATH is not None:
    with open(EXISTING_VOCAB_PATH) as f:
        vocab = json.load(f)
    print(f"Loaded existing vocab with {len(vocab)} entries")
else:
    vocab = {gene: i for i, gene in enumerate([CLS_TOKEN, PAD_TOKEN] + gene_names)}
    print(f"Built vocab with {len(vocab)} entries (2 special + {len(gene_names)} genes)")

with open(data_path.vocab_path, "w") as f:
    json.dump(vocab, f)

## 3. Convert expression matrix to chunked h5ad files

Each chunk becomes one h5ad file with:
- `.X`: sparse expression matrix (samples × genes)
- `.var_names`: gene symbols
- `.var["_cf_gene_id"]`: vocab indices per gene
- `.obs`: metadata columns from `metadata.csv`

These h5ad files are an intermediate step consumed by `SingleCellCollection`.

In [ ]:
h5ad_dir = OUTPUT_DIR / "h5ads"
h5ad_dir.mkdir(parents=True, exist_ok=True)

# Map gene names to vocab IDs; drop genes not in vocab
gene_ids = np.array([vocab.get(g, -1) for g in gene_names])
valid_mask = gene_ids >= 0
valid_gene_names = [g for g, v in zip(gene_names, valid_mask) if v]
valid_gene_ids = gene_ids[valid_mask]
valid_col_indices = np.where(valid_mask)[0]

print(f"Genes in vocab: {len(valid_gene_names)}/{len(gene_names)}")

# Use sample_id as index if available, else generate one
if "sample_id" in metadata.columns:
    metadata = metadata.set_index("sample_id", drop=False)

with h5py.File(EXPRESSION_H5, "r") as f:
    expr_ds = f[expr_key]
    n_total = expr_ds.shape[0]
    n_chunks = (n_total + CHUNK_SIZE - 1) // CHUNK_SIZE

    for chunk_idx in range(n_chunks):
        start = chunk_idx * CHUNK_SIZE
        end = min(start + CHUNK_SIZE, n_total)

        # Read chunk and select valid genes
        X_dense = expr_ds[start:end, :][:, valid_col_indices]
        X_sparse = sp.csr_matrix(X_dense.astype(np.float32))

        obs_chunk = metadata.iloc[start:end][OBS_COLUMNS].copy()
        obs_chunk.index = obs_chunk.index.astype(str)

        var = pd.DataFrame({GENE_ID: valid_gene_ids}, index=valid_gene_names)
        var[GENE_ID] = var[GENE_ID].astype(int)

        adata = ad.AnnData(X=X_sparse, obs=obs_chunk, var=var)

        fname = f"bulk_chunk_{chunk_idx:04d}.h5ad"
        adata.write_h5ad(h5ad_dir / fname)
        print(f"  [{chunk_idx+1}/{n_chunks}] wrote {fname}  ({end - start} samples)")

print(f"\nCreated {n_chunks} h5ad files in {h5ad_dir}")

## 4. Build memory-mapped dataset via SingleCellCollection

Same flow as the single-cell pipeline: load h5ad files → flatten to `mem.map/`.

In [ ]:
memmaps = SingleCellCollection(data_path.data_dir / "tmp")
memmaps.load_h5ad_multi(str(h5ad_dir), max_workers=12)

memmaps.flatten(data_path.memmap_path, destroy_on_copy=True)

# Remove duplicate metadata file that causes issues (same as single-cell pipeline)
dup_parquet = data_path.memmap_path / "features/dataframe_00.parquet"
if os.path.isfile(dup_parquet):
    os.remove(dup_parquet)

print(f"Memory-mapped dataset created at {data_path.memmap_path}")

## 5. Build obs.parquet and mapping.json

Encode selected metadata columns as categorical integers, matching the single-cell pipeline format.

In [ ]:
def convert_columns_to_categorical_with_mapping(df):
    """Encode each column as categorical integers and return the mapping."""
    df_copy = df.copy()
    category_mappings = {}
    df_categorical = pd.DataFrame(index=df.index)

    for column in df.columns:
        df_copy[column] = df_copy[column].astype("category")
        category_mappings[column] = dict(
            zip(
                df_copy[column].cat.categories,
                range(len(df_copy[column].cat.categories)),
            )
        )
        df_categorical[column] = df_copy[column].cat.codes

    return df_categorical, category_mappings


# Use the full metadata (same row order as the expression matrix)
obs = metadata[OBS_COLUMNS].copy().reset_index(drop=True)
obs, mapping = convert_columns_to_categorical_with_mapping(obs)

obs.to_parquet(data_path.obs_path)

with data_path.mapping_path.open("w") as f:
    json.dump(mapping, f, indent=4)

print(f"obs.parquet: {obs.shape}")
print(f"mapping.json keys: {list(mapping.keys())}")
for k, v in mapping.items():
    print(f"  {k}: {len(v)} categories")

## 6. Validate & clean up

In [ ]:
from bionemo.scdl.io.single_cell_memmap_dataset import SingleCellMemMapDataset

# Validate the output directory has all required files
assert data_path.validate(), f"Missing files in {OUTPUT_DIR}"

# Quick sanity check: load the memmap and verify row count matches metadata
memmap = SingleCellMemMapDataset(data_path.memmap_path)
obs_check = pd.read_parquet(data_path.obs_path)

print(f"Memmap rows:  {memmap.number_of_rows()}")
print(f"Obs rows:     {obs_check.shape[0]}")
assert memmap.number_of_rows() == obs_check.shape[0], "Row count mismatch!"

# Clean up intermediate h5ad files
shutil.rmtree(h5ad_dir)
print(f"\nCleaned up intermediate h5ads at {h5ad_dir}")

print(f"\nDone! Pipeline-ready data at: {OUTPUT_DIR}")
print(f"  vocab.json   ({len(vocab)} genes)")
print(f"  obs.parquet  ({obs_check.shape[0]} samples × {obs_check.shape[1]} columns)")
print(f"  mapping.json ({len(mapping)} columns)")
print(f"  mem.map/     ({memmap.number_of_rows()} rows)")